# Query transformation

Sometimes the user's literal question isn't the best *retrieval* query. Four classic transformations help:

1. **HyDE** — generate a hypothetical answer and embed *that* for retrieval. The answer's vocabulary matches the corpus better than the question's does.
2. **Multi-query** — generate N paraphrases of the question, retrieve for each, union the results. Cheap recall boost.
3. **Step-back** — generalize the question to a broader concept before retrieving. Useful when the corpus answers the concept, not the specific.
4. **Decomposition** — split a multi-hop question into atomic sub-questions, retrieve for each, recombine.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import numpy as np
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/05-query-transformation'
DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa() if q.source_ids}
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)

def retrieve(text: str, k: int = 3) -> list[str]:
    qv = hash_embed([text], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [doc_ids[i] for i in idx]

## 1) HyDE
Generate a hypothetical answer to the question, then embed *that*. The answer's vocabulary tends to align with the corpus better than the question's does.

In [3]:
def hyde(q: str) -> str:
    return complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=(
                'Write a short, plausible-sounding passage (2-3 sentences) that '
                'ANSWERS the following research question. Do not hedge. The passage '
                'will be used purely as an embedding query — its content does not '
                'need to be true.')),
            Message(role='user', content=q),
        ],
    ).content

q = QA['q01'].question
fake = hyde(q)
print('HYDE:', fake[:200], '...')
print('  vanilla retrieved:', retrieve(q))
print('  HyDE retrieved   :', retrieve(fake))

HYDE: RA-MoE reduces p99 decode latency by 38% relative to standard learned routing. The mechanism is a router-logit bias toward experts whose recent activation minimizes inter-device communication. ...
  vanilla retrieved: ['synth-001', 'synth-008', 'synth-004']
  HyDE retrieved   : ['synth-001', 'synth-008', 'synth-002']


## 2) Multi-query
N paraphrases, N retrievals, union the result lists. Cheap recall boost.

In [4]:
def multi_query(q: str) -> list[str]:
    raw = complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=(
                'Rewrite the user\'s question as 3 SEMANTICALLY EQUIVALENT alternative '
                'phrasings. One per line, no numbering, no extra prose.')),
            Message(role='user', content=q),
        ],
    ).content
    return [line.strip() for line in raw.splitlines() if line.strip()]

def retrieve_union(qs: list[str], k: int = 3) -> list[str]:
    seen: dict[str, int] = {}
    for q in qs:
        for rank, doc_id in enumerate(retrieve(q, k=k)):
            seen[doc_id] = min(seen.get(doc_id, 1000), rank)
    return sorted(seen, key=lambda d: seen[d])

q = QA['q05'].question
alts = multi_query(q)
for a in alts:
    print(' -', a)
print('vanilla :', retrieve(q))
print('multi-q :', retrieve_union([q, *alts])[:5])

 - What attention mechanism does HA-Retrieve use for long documents?
 - How does HA-Retrieve handle million-token inputs?
 - What's the architecture of HA-Retrieve's attention?
vanilla : ['synth-006', 'synth-001', 'synth-003']
multi-q : ['synth-006', 'synth-002', 'synth-001', 'synth-005', 'synth-003']


## 3) Step-back
Generalize before retrieving. Especially useful when the corpus stores conceptual answers rather than entity-specific ones.

In [5]:
def step_back(q: str) -> str:
    return complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=(
                'Given a SPECIFIC technical question, rewrite it as a MORE GENERAL '
                'question about the underlying concept. Reply with just the '
                'generalized question.')),
            Message(role='user', content=q),
        ],
    ).content.strip()

q = QA['q01'].question
general = step_back(q)
print('SPECIFIC:', q)
print('GENERAL :', general)

SPECIFIC: By how much does RA-MoE reduce p99 decode latency compared to standard learned routing?
GENERAL : How do routing strategies affect inference latency in mixture-of-experts models?


## 4) Decomposition
Multi-hop questions can be split into independent atomic sub-questions. Retrieve for each, then recombine in the final prompt.

In [6]:
def decompose(q: str) -> list[str]:
    raw = complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=(
                'Decompose the user\'s multi-hop question into 2-3 atomic '
                'sub-questions, one per line. Each sub-question should be '
                'answerable from a SINGLE document.')),
            Message(role='user', content=q),
        ],
    ).content
    return [line.strip() for line in raw.splitlines() if line.strip()]

from shared.loaders import load_golden_qa
full_qa = {q.id: q for q in load_golden_qa()}
q = full_qa['q23'].question
subs = decompose(q)
for s in subs:
    print(' -', s, '->', retrieve(s, k=2))

 - How many parameters does the RA-MoE language model have? -> ['synth-001', 'synth-008']
 - How many experts does RA-MoE use? -> ['synth-001', 'synth-009']
 - What is RA-MoE's routing strategy? -> ['synth-001', 'synth-010']


## When to reach for each

| Transformation | Best for | Cost |
|---|---|---|
| HyDE | Question vocab differs from corpus vocab | 1 LLM call |
| Multi-query | Recall-bound retrieval; cheap embeddings | 1 LLM + N retrievals |
| Step-back | Concept-level corpora; FAQ-style | 1 LLM call |
| Decomposition | Multi-hop questions (joins across docs) | 1 LLM + N retrievals + 1 synth |

Forward reference: combine **multi-query + RRF** for the strongest first stage, then rerank — that's what production stacks like Anthropic's Contextual Retrieval do.